# Week 14 — Python Solution Lab
## Capstone Synthesis

**Companion to `notebooks/Week_14.ipynb`.** This notebook contains *fully worked Python
solutions* to selected problems from that week's problem set — one at **each difficulty level**.

Every solution follows the course's core workflow:

> **Diagram → Principle → Equation → Predict → Verify**

The markdown cell states the problem, identifies the governing principle, and gives the **hand
prediction you should make before running anything**. The code cell then computes the result and
*verifies* it — typically by a second independent method (energy vs. forces, symbolic vs.
numerical, closed form vs. simulation) — and includes `assert` checks against the known answer.

### How to use this notebook

1. **Attempt the problem in `Week_14.ipynb` first.** These solutions are worth very little
   if you read them before trying.
2. Make the hand prediction. Write it down.
3. Run the code cell and compare.
4. **Change a number and re-run.** Every solution is written so that the parameters sit at the
   top; the sweeps and plots update automatically. Ask "what if the mass doubled?" and answer it
   in ten seconds.

### Why the code looks like this

These are not minimal answer-generators. Each one demonstrates something Python does that hand
algebra cannot: parameter sweeps, root-finding, numerical integration, symbolic differentiation,
or a cross-check to machine precision. The physics is the point; the code is how we prove the
physics is right.

---


### Solutions in this notebook

| Level | Problem | Topic | Python technique |
|---|---|---|---|
| **L1 · Basic** | `P2` | Energy Conservation on a Frictionless Incline | one system, three principles, one answer |
| **L2 · Intermediate** | `P7` | Counting Oscillations of a Damped Block | peak-counting: turning-point decay vs sinusoidal envelope |
| **L3 · Challenge** | `P9` | Rolling Bodies Climbing an Incline | energy closure + simulation, course synthesis |

---

## L1 · Basic — P2: Energy Conservation on a Frictionless Incline

> **Problem (Week_14.ipynb, L1 — P2).** A $5.0$ kg block slides down a $30^\circ$ frictionless
> incline from a height of $2.0$ m. Find its speed at the bottom using energy conservation.

**Diagram → Principle.** No friction means mechanical energy is conserved. All the gravitational
PE becomes KE.

**Equation.** $mgh = \tfrac12mv^2 \Rightarrow v = \sqrt{2gh}$.

**Hand prediction.** $v = \sqrt{2(9.81)(2.0)} = 6.26$ m/s.

**What Python adds.** The capstone lesson of the whole course: **the same system, solved by three
different principles, must give the same answer.** We solve it by energy, by $F = ma$ plus
kinematics along the incline, and by direct numerical integration of the equation of motion — and
show all three agree. We also confirm that neither the mass nor the incline angle affects the
final speed, only the height.

In [ ]:
# ═══ W14 · L1 · P2 — One system, three principles, one answer ═══
import numpy as np
from scipy.integrate import solve_ivp

# --- MODEL --------------------------------------------------------------
m, h, theta_deg, g = 5.0, 2.0, 30.0, 9.81
theta = np.radians(theta_deg)
d = h/np.sin(theta)                     # length along the incline

# --- ROUTE 1: energy conservation ---------------------------------------
v_energy = np.sqrt(2*g*h)
print(f"ROUTE 1 -- energy:      m g h = (1/2) m v^2")
print(f"  PE released = {m*g*h:.4f} J  ->  v = sqrt(2 g h) = {v_energy:.4f} m/s")

# --- ROUTE 2: Newton + kinematics along the slope -----------------------
a = g*np.sin(theta)
v_newton = np.sqrt(2*a*d)
print(f"\nROUTE 2 -- Newton:      a = g sin(theta) = {a:.4f} m/s^2 along a {d:.4f} m slope")
print(f"  v = sqrt(2 a d) = {v_newton:.4f} m/s")

# --- ROUTE 3: integrate the equation of motion --------------------------
def hit_bottom(t, y): return y[0] - d
hit_bottom.terminal, hit_bottom.direction = True, 1
sol = solve_ivp(lambda t, y: [y[1], a], [0, 10], [0.0, 0.0],
                events=hit_bottom, rtol=1e-12, atol=1e-14)
v_sim = sol.y_events[0][0][1]
t_sim = sol.t_events[0][0]
print(f"\nROUTE 3 -- simulation:  reaches the bottom at t = {t_sim:.4f} s")
print(f"  v = {v_sim:.4f} m/s")

print(f"\nagreement: max spread = {max(v_energy, v_newton, v_sim) - min(v_energy, v_newton, v_sim):.2e} m/s")
assert abs(v_energy - v_newton) < 1e-9 and abs(v_energy - v_sim) < 1e-6

# --- What the answer does and does NOT depend on ------------------------
print(f"\n  the final speed depends ONLY on the height:")
print(f"  {'mass':>8s} {'angle':>8s} {'slope len':>11s} {'time (s)':>10s} {'v (m/s)':>9s}")
for mt in (0.5, 5.0, 500.0):
    for th_d in (15.0, 30.0, 90.0):
        th = np.radians(th_d)
        dd = h/np.sin(th); aa = g*np.sin(th)
        print(f"  {mt:8.1f} {th_d:8.0f} {dd:11.4f} {np.sqrt(2*dd/aa):10.4f} "
              f"{np.sqrt(2*aa*dd):9.4f}")
print("  -> v is identical every time; only the TIME taken changes with the angle.")
print("     (theta = 90 deg is simply free fall, and it arrives fastest.)")

# --- CHECK --------------------------------------------------------------
assert abs(v_energy - 6.2642) < 1e-3
print(f"\n[OK] Matches textbook answer: v = {v_energy:.2f} m/s")

## L2 · Intermediate — P7: Counting Oscillations of a Damped Block

> **Problem (Week_14.ipynb, L2 — P7).** A $0.30$ kg block on a horizontal spring ($k = 75$ N/m)
> has damping $b = 0.60$ N·s/m. Displaced $0.10$ m and released. Find the number of oscillations
> before the amplitude drops to $0.01$ m.

**Diagram → Principle.** The successive turning-point amplitudes decay as $A_0e^{-\gamma t}$.
Solve for the time at which that curve reaches the target, then convert to damped periods.

> **"Amplitude" needs pinning down.** For release from rest,
> $x = Ce^{-\gamma t}\cos(\omega_dt-\phi)$ with $C = A_0\sqrt{1+(\gamma/\omega_d)^2}$. Setting
> $\dot x = 0$ gives $\omega_dt = n\pi$, and at those instants $|x| = A_0e^{-\gamma t}$ exactly.
> So $A_0e^{-\gamma t}$ is the **turning-point decay curve**, threading the real extrema; the
> **sinusoidal envelope** is the slightly larger $Ce^{-\gamma t}$. Here
> $C = 0.1002006$ m versus $A_0 = 0.10$ m — a $0.20\%$ gap, which moves the crossing time from
> $2.30259$ s to $2.30459$ s ($5.783$ vs $5.788$ periods). We report the turning-point figure,
> because "the amplitude of the oscillation" means the size of the actual swings.

**Equation.** $\gamma = b/2m$, $t = \dfrac{\ln(A_0/A)}{\gamma}$, $n = t/T_d$.

**Hand prediction.** $\gamma = 1.0$ s⁻¹, $t = \ln(10) = 2.303$ s, $\omega_d \approx 15.78$ rad/s,
$T_d = 0.398$ s, so $n \approx 5.78$.

> **Two readings of "number of oscillations" — quote both.** The turning-point curve
> crosses $0.01$ m at
> $t = 2.303$ s, which is **5.78 periods**. But oscillations are counted in whole cycles, and the
> peak amplitudes go $\ldots$ $0.01366$ m (5th peak), $0.00917$ m (6th) — so the **6th peak is the
> first one below $0.01$ m**. Say "the amplitude falls below $0.01$ m after $5.78$ periods, i.e.
> during the 6th oscillation" and no student can argue the mark scheme.

> ⚠️ The key printed in `Week_14.ipynb` gives $n \approx 3.66$. The cell below derives $5.78$ from
> the stated numbers and cross-checks it against a direct simulation.

**What Python adds.** A closed-form answer and a simulation that literally **counts the peaks** of
the decaying oscillation. When a hand result is disputed, counting the actual maxima of the
integrated solution settles it — and it also settles which of the two decay curves is the
relevant one, by showing the peaks land on $A_0e^{-\gamma t}$ to seven decimals.

In [ ]:
# ═══ W14 · L2 · P7 — Counting damped oscillations: formula vs peak-counting ═══
import numpy as np
from scipy.integrate import solve_ivp
from scipy.signal import find_peaks
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, k, b   = 0.30, 75.0, 0.60
A0, A_tgt = 0.10, 0.01

gamma = b/(2*m)
w0    = np.sqrt(k/m)
wd    = np.sqrt(w0**2 - gamma**2)
Td    = 2*np.pi/wd

print(f"gamma   = b/2m = {gamma:.4f} 1/s")
print(f"omega_0 = {w0:.4f} rad/s,  omega_d = {wd:.4f} rad/s")
print(f"T_d     = {Td:.5f} s   (damping ratio zeta = {gamma/w0:.4f}, very lightly damped)")

# --- PREDICT: decay of the turning-point amplitudes ---------------------
t_decay = np.log(A0/A_tgt)/gamma
n_osc   = t_decay/Td
print(f"\nA0 e^(-gamma t) = A_target  ->  t = ln(A0/A)/gamma = ln({A0/A_tgt:.0f})/{gamma:.1f}"
      f" = {t_decay:.5f} s")
print(f"number of oscillations n = t / T_d = {n_osc:.4f}")

# --- VERIFY by simulating and counting actual peaks ---------------------
f = lambda t, y: [y[1], (-k*y[0] - b*y[1])/m]
sol = solve_ivp(f, [0, t_decay*1.6], [A0, 0.0], rtol=1e-12, atol=1e-14, dense_output=True)
tt = np.linspace(0, t_decay*1.6, 400000)
xx = sol.sol(tt)[0]

pk, _ = find_peaks(xx)
t_pk, x_pk = tt[pk], xx[pk]
print(f"\nsimulation: found {len(t_pk)} positive peaks before t = {t_decay*1.6:.2f} s")
print(f"  {'peak #':>7s} {'t (s)':>9s} {'amplitude (m)':>15s}")
for i, (tp, xp_) in enumerate(zip(t_pk[:9], x_pk[:9]), start=1):
    print(f"  {i:7d} {tp:9.4f} {xp_:15.6f}")

# the peak whose amplitude first falls below the target
below = np.argmax(x_pk < A_tgt)
print(f"\n  amplitude first falls below {A_tgt} m at peak #{below+1}, t = {t_pk[below]:.4f} s")
print(f"  measured decay time (interpolating through the peaks): ", end="")
t_meas = np.interp(-np.log(A_tgt), -np.log(x_pk), t_pk)   # -log(x) rises with t
print(f"{t_meas:.4f} s  vs formula {t_decay:.4f} s")
assert abs(t_meas - t_decay) < 0.05

# measured period from successive peaks
Td_meas = np.mean(np.diff(t_pk))
print(f"  measured T_d from peak spacing = {Td_meas:.5f} s  vs formula {Td:.5f} s")
assert abs(Td_meas - Td) < 1e-3
print(f"  -> n = {t_meas/Td_meas:.3f} oscillations, confirming the closed form.")

# --- Reconcile with the printed key -------------------------------------
print(f"\nANSWER-KEY NOTE: the key prints n ~ 3.66.")
print(f"  From the stated m, k, b: gamma = {gamma:.2f}, T_d = {Td:.4f} s, and reaching")
print(f"  one tenth of the amplitude takes ln(10)/gamma = {t_decay:.3f} s = {n_osc:.2f} periods.")
print(f"  Both the closed form and direct peak-counting give {n_osc:.2f}. Use n ~ {n_osc:.2f}.")

# --- Plot ---------------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tt, xx, color="#1565c0", lw=1.1, label="x(t)")
ax.plot(tt,  A0*np.exp(-gamma*tt), "--", color="#e65100", lw=2,
        label="turning-point curve $A_0e^{-\\gamma t}$")
ax.plot(tt, -A0*np.exp(-gamma*tt), "--", color="#e65100", lw=2)
ax.plot(t_pk, x_pk, "o", color="#2e7d32", ms=4, label="counted peaks")
ax.axhline(A_tgt, color="crimson", ls=":", lw=2, label=f"target {A_tgt} m")
ax.axvline(t_decay, color="grey", ls=":", lw=1.5)
ax.set_xlabel("t (s)"); ax.set_ylabel("x (m)")
ax.set_title(f"W14 P7 — amplitude reaches {A_tgt} m after {n_osc:.2f} oscillations")
ax.grid(alpha=.3); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(gamma - 1.0) < 1e-12
assert abs(t_decay - 2.3026) < 1e-3
assert abs(n_osc - 5.782) < 5e-3, f"n = {n_osc}"
# --- turning-point curve vs sinusoidal envelope -------------------------
C_env = A0*np.sqrt(1 + (gamma/wd)**2)
t_sin = np.log(C_env/A_tgt)/gamma
print(f"\nWHICH DECAY CURVE? (they differ by {100*(C_env/A0 - 1):.2f}% here)")
print(f"  turning-point curve  A0 e^-gt, prefactor {A0:.7f} m  -> "
      f"t = {t_decay:.5f} s, n = {t_decay/Td:.4f}")
print(f"  sinusoidal envelope  C  e^-gt, prefactor {C_env:.7f} m  -> "
      f"t = {t_sin:.5f} s, n = {t_sin/Td:.4f}")
print(f"  C = A0 sqrt(1 + (gamma/w_d)^2) = {A0} * sqrt(1 + ({gamma}/{wd:.4f})^2)")
print(f"  The measured peaks above lie on the TURNING-POINT curve, so that is the")
print(f"  one to quote: 'the amplitude of the oscillation' means the actual swings.")
err_tp = np.max(np.abs(x_pk - A0*np.exp(-gamma*t_pk)))
print(f"  check: simulated peaks vs A0 e^-gt, max error {err_tp:.1e} m")
assert err_tp < 1e-6, "peaks must lie on the turning-point curve"
assert abs(t_sin - 2.30459) < 1e-4 and C_env > A0

# --- state BOTH readings so the mark scheme is unambiguous --------------
print(f"\nTWO WAYS TO SAY IT (quote both to students):")
print(f"  continuous : the turning-point curve reaches {A_tgt} m after {n_osc:.2f} periods "
      f"(t = {t_decay:.3f} s)")
print(f"  by cycles  : peak {below} is {x_pk[below-1]*1000:.2f} mm (still above), "
      f"peak {below+1} is {x_pk[below]*1000:.2f} mm (first below)")
print(f"               -> it drops below {A_tgt} m DURING the {below+1}th oscillation")
assert below + 1 == 6, f"expected the 6th peak to be the first below, got {below+1}"

print(f"\n[OK] n = {n_osc:.2f} periods (t = {t_decay:.3f} s, T_d = {Td:.4f} s); "
      f"equivalently, during the 6th cycle. Confirmed by peak counting.")

## L3 · Challenge — P9: Rolling Bodies Climbing an Incline

> **Problem (Week_14.ipynb, L3 — P9).** A solid sphere ($m = 3.0$ kg, $R = 0.08$ m) rolls
> without slipping up a $25^\circ$ incline from $5.0$ m/s. (a) How far does it travel before
> stopping? (b) Same for a hollow sphere. (c) Explain the difference with energy methods.

**Diagram → Principle.** The capstone integrates the whole course: rolling constraint (W9),
rotational inertia (W9), energy conservation (W6), and inclines (W4). A rolling body carries
**both** translational and rotational kinetic energy, and both must be converted to potential
energy before it stops.

**Equation.** $\tfrac12mv^2(1+\beta) = mgd\sin\theta \Rightarrow d = \dfrac{v^2(1+\beta)}{2g\sin\theta}$.

**Hand prediction.** Solid ($\beta = 2/5$): $d = 25(1.4)/(2\cdot9.81\cdot0.4226) = 4.22$ m.
Hollow ($\beta = 2/3$): $d = 5.02$ m.

> ⚠️ The key prints $d_{\rm solid} = 4.33$ m. With $\sin 25^\circ = 0.42262$ and $g = 9.81$ the
> correct value is $4.22$ m; $4.33$ corresponds to $g = 9.55$ or $\sin\theta = 0.412$. The cell
> below shows the arithmetic explicitly.

**What Python adds.** We confirm the energy result by integrating the rolling equation of motion
to the turning point, verify the rolling constraint $v = \omega R$ holds throughout, and show the
mass and radius cancel — the hollow sphere goes further **purely** because more of its energy is
stored in rotation.

In [ ]:
# ═══ W14 · L3 · P9 — Rolling up an incline: the whole course in one problem ═══
import numpy as np
from scipy.integrate import solve_ivp
import matplotlib.pyplot as plt

# --- MODEL --------------------------------------------------------------
m, R, v0, theta_deg, g = 3.0, 0.08, 5.0, 25.0, 9.81
theta = np.radians(theta_deg)

def climb(beta):
    """Distance along the incline before a rolling body stops."""
    return v0**2*(1 + beta) / (2*g*np.sin(theta))

b_sol, b_hol = 2/5, 2/3
d_sol, d_hol = climb(b_sol), climb(b_hol)

print(f"incline {theta_deg:.0f} deg, sin(theta) = {np.sin(theta):.5f}, v0 = {v0} m/s\n")
print(f"(a) solid sphere  (beta = 2/5): d = v0^2(1+beta)/(2 g sin th)")
print(f"      = {v0**2:.1f} * {1+b_sol:.4f} / (2 * {g} * {np.sin(theta):.5f}) = {d_sol:.4f} m")
print(f"(b) hollow sphere (beta = 2/3): d = {d_hol:.4f} m")
print(f"    the hollow sphere climbs {d_hol - d_sol:.4f} m further "
      f"({100*(d_hol/d_sol - 1):.1f}% more)")

# --- (c) the energy explanation, in numbers -----------------------------
print(f"\n(c) energy accounting at the moment of launch (per body, {m} kg at {v0} m/s):")
print(f"  {'body':16s} {'KE_trans':>10s} {'KE_rot':>9s} {'total':>9s} {'rot share':>11s} {'d (m)':>8s}")
for nm, beta, d in (("solid sphere", b_sol, d_sol), ("hollow sphere", b_hol, d_hol)):
    KE_t = 0.5*m*v0**2
    KE_r = 0.5*beta*m*v0**2
    print(f"  {nm:16s} {KE_t:10.4f} {KE_r:9.4f} {KE_t+KE_r:9.4f} "
          f"{100*KE_r/(KE_t+KE_r):10.1f}% {d:8.4f}")
    assert abs((KE_t + KE_r) - m*g*d*np.sin(theta)) < 1e-9, "energy must close"
print("  Both start at the same SPEED, but the hollow sphere carries more total energy")
print("  because more of it is locked in rotation -- so it must climb further to shed it.")

# --- VERIFY by integrating the rolling equation of motion ---------------
print(f"\n  simulation check (a = -g sin(theta)/(1+beta), stop when v = 0):")
for nm, beta, d in (("solid ", b_sol, d_sol), ("hollow", b_hol, d_hol)):
    a = -g*np.sin(theta)/(1 + beta)
    stop = lambda t, y: y[1]
    stop.terminal, stop.direction = True, -1
    s = solve_ivp(lambda t, y: [y[1], a], [0, 5], [0.0, v0],
                  events=stop, rtol=1e-12, atol=1e-14)
    d_sim, t_sim = s.y_events[0][0][0], s.t_events[0][0]
    print(f"    {nm}: a = {a:7.4f} m/s^2, stops at t = {t_sim:.4f} s, d = {d_sim:.4f} m "
          f"(formula {d:.4f})")
    assert abs(d_sim - d) < 1e-6

# --- Rolling constraint holds throughout --------------------------------
a_s = -g*np.sin(theta)/(1 + b_sol)
ts  = np.linspace(0, v0/abs(a_s), 200)
v_t = v0 + a_s*ts
w_t = v_t/R
print(f"\n  rolling constraint v = omega R checked at 200 instants: "
      f"max error {np.max(np.abs(v_t - w_t*R)):.2e} m/s")
assert np.allclose(v_t, w_t*R)

# --- Mass and radius cancel ---------------------------------------------
print(f"\n  neither m nor R appears in d = v0^2(1+beta)/(2 g sin th):")
for mt, Rt in ((0.03, 0.005), (3.0, 0.08), (300.0, 1.2)):
    print(f"    m = {mt:7.2f} kg, R = {Rt:5.3f} m -> d_solid = {climb(b_sol):.4f} m")

# --- Reconcile with the printed key -------------------------------------
print(f"\nANSWER-KEY NOTE: the key prints d_solid = 4.33 m; the formula gives {d_sol:.3f} m.")
print(f"  4.33 m would require g sin(theta) = {v0**2*(1+b_sol)/(2*4.33):.4f}, i.e. "
      f"g = {v0**2*(1+b_sol)/(2*4.33)/np.sin(theta):.3f} m/s^2 instead of 9.81.")
print(f"  With the problem's stated values, use d_solid = {d_sol:.2f} m, "
      f"d_hollow = {d_hol:.2f} m.")

# --- Plot ---------------------------------------------------------------
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.6, 3.9))
for nm, beta, col in (("solid ($\\beta$=2/5)", b_sol, "#2e7d32"),
                      ("hollow ($\\beta$=2/3)", b_hol, "#e65100"),
                      ("sliding ($\\beta$=0)", 0.0, "grey")):
    a = -g*np.sin(theta)/(1 + beta)
    t = np.linspace(0, v0/abs(a), 300)
    ax1.plot(t, v0*t + 0.5*a*t**2, lw=2, color=col, label=nm)
ax1.set_xlabel("t (s)"); ax1.set_ylabel("distance up the incline (m)")
ax1.set_title("how far each body gets"); ax1.grid(alpha=.3); ax1.legend(fontsize=8)

betas = np.linspace(0, 1.2, 300)
ax2.plot(betas, climb(betas), color="#1565c0", lw=2.5)
for nm, beta, col in (("solid", b_sol, "#2e7d32"), ("hollow", b_hol, "#e65100"),
                      ("hoop", 1.0, "crimson")):
    ax2.plot(beta, climb(beta), "o", color=col, ms=9, zorder=5,
             label=f"{nm}: {climb(beta):.2f} m")
ax2.set_xlabel("shape factor $\\beta = I/mR^2$"); ax2.set_ylabel("climbing distance (m)")
ax2.set_title("more rotational inertia -> climbs further")
ax2.grid(alpha=.3); ax2.legend(fontsize=8)
plt.suptitle(f"W14 P9 — rolling up a {theta_deg:.0f} deg incline at {v0} m/s", y=1.03)
plt.tight_layout(); plt.show()

# --- CHECK --------------------------------------------------------------
assert abs(d_sol - 4.2213) < 1e-3, f"d_solid = {d_sol}"
assert abs(d_hol - 5.0254) < 1e-3, f"d_hollow = {d_hol}"
assert d_hol > d_sol
print(f"\n[OK] d_solid = {d_sol:.2f} m, d_hollow = {d_hol:.2f} m "
      f"(energy closure and simulation both verified).")

---

## Self-check

**Every code cell above contains one or more `assert` checks.** If you run the whole notebook top
to bottom without an `AssertionError`, all of the numerical checks on this page have passed.
(The asserts sit just before each cell's closing summary, so the last thing you see is a printed
result — not the check itself.)

**Now transfer the skill.** Pick one unsolved problem from `Week_14.ipynb` at the level you
found hardest, and write the same five-part structure for it:

```python
# --- MODEL:   parameters at the top, with units in comments
# --- PREDICT: the closed-form answer
# --- VERIFY:  a SECOND, independent route to the same number
# --- CHECK:   assert against your hand prediction
```

The verify step is the one that matters. A result you have only computed one way is a result you
have not checked.
